<a href="https://colab.research.google.com/github/ProfessorPatrickSlatraigh/CIS3120-BMWB/blob/main/cis3120_module16_WebScraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 16: Web Scraping


| | |  
|:---|:---|  
| **Instructor** | Patrick Slattery |  
| **Estimated time** | 90 minutes (two class meetings of 45 minutes each) |   
| <i>Copy avilable at:</i>| [<i>bit.ly/cis3120_module16</i>](https://bit.ly/cis3120_module16)|  
  
This notebook is for lessons in Module 16. It contains
examples from the reading page, demonstrations the instructor performs live, and class
exercises. The <b>Web Scraping</b> topics include use of `requests`, <b>BeautifulSoup</b>, and `pd.read_html()`.  


## Module Overview

Web scraping is the technique of extracting structured data from web pages by
retrieving their HTML and parsing the elements directly, rather than receiving
structured data from a Web API. This module teaches scraping responsibly and
skillfully, and teaches recognition of when scraping is the wrong tool and an
API would serve better.

The module is the practical complement to the Web APIs module that precedes it.
Both rely on the same HTTP request-response pattern and use the same Python
library, `requests`. The substantive difference is the response payload: APIs
return JSON, which the `requests` library parses into a Python dictionary in a
single line; scraped pages return HTML, which requires a parsing step using
BeautifulSoup or `pandas.read_html()`.

### Learning Outcomes

After completing Module 16, a student should be able to do each of the following:

1. Distinguish situations in which web scraping is the appropriate technique from situations in which a Web API would be preferred, and articulate the reasoning.
2. Use the `requests` library to retrieve HTML from a URL with appropriate headers, error handling, and timeout discipline.
3. Parse retrieved HTML using BeautifulSoup to extract specific elements identified by tag, class, attribute, or CSS selector.
4. Extract a tabular HTML region into a Pandas DataFrame using `pandas.read_html()`, including handling the type-coercion and missing-value behaviors that result from real-world tables.
5. Recognize and respond to common defensive-extraction scenarios: missing cells, mixed-type columns, special-string tokens in numeric columns, and inconsistent formatting across rows.
6. Identify yourself as a courteous client through user-agent headers and respect site-level signals such as `robots.txt`.
7. Combine fetch, parse, and DataFrame construction into a complete extract pipeline that produces analyzable structured data from a published web page.



---



### Example 1 · Viewing The Target Site
  
Examples in this notebook work with a website writtent in WordPress by the professor.  As you will see, WordPress sites offer default APIs, including an API that serves web page content as `JSON`. You can view the website `https://datarollup.io` with your browser by double-clicking on the URL or typing the URL into your search bar.

```
https://datarollup.io  
```



---



## Setup

The libraries used in this module are all installed in Google Colab by default.
No installation step is required. The cell below imports them and defines the
 `HEADERS` dictionary used on every outbound request to `datarollup.io`.

The User-Agent string identifies the client as a Baruch College class lab and
provides a contact email so any administrator who reviews server logs can reach
the course author. This is the courteous default for any outbound HTTP request
and is a teaching point in its own right.

<font color=blue>⚠️ Be sure to replace `fname.lname@baruch.cuny.edu` in the following code with your email address - <i>a burner email is okay.</i></font>

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO

HEADERS = {
    "User-Agent": "DataRollup-Scraping-Lab/1.0 "
                  "(Baruch College; contact: fname.lname@baruch.cuny.edu)"
}

print("Setup complete. Libraries imported and HEADERS configured.")

---

# Class 1 · Foundations and Static Page Extraction

Class 1 establishes the conceptual framing (when scraping is and is not the
right tool), the HTTP and HTML preliminaries (the request-response model and
the structure of an HTML document), the `requests` library applied to HTML
retrieval, BeautifulSoup as the  parser, and a closing demonstration
on the simplest possible HTML pages.

## Topic 1.1 · When Scraping Is and Is Not the Right Tool

Web scraping is a fallback technique. It is the right tool when a Web API does
not exist for the data needed, when an API exists but does not expose the
specific fields needed, or when the data is published only as rendered HTML.
It is the wrong tool when an API exists and serves the same data more cleanly,
since scraping is more brittle, more bandwidth-intensive, and more legally
ambiguous than API access.

> **API-First Principle.** The site we are about to scrape exposes a complete
> JSON API at `https://datarollup.io/wp-json/wp/v2/pages` listing every
> published page. A thoughtful analyst checks for the API before reaching for
> scraping tools. The example below demonstrates this principle on the same
> site that will be scraped in subsequent topics.

### Key Terms

- **API-first principle:** always check whether a documented API exists before deciding to scrape.
- **Scraping versus crawling:** scraping retrieves and parses a known page; crawling traverses a site by following links to discover pages. The two are related but distinct.
- **Terms of service:** the legal contract between a website operator and a visitor; some operators explicitly prohibit automated retrieval.
- **Robots Exclusion Standard:** the convention by which a site indicates which paths are open to automated access via `/robots.txt`.

### Common Misconception

Students sometimes treat scraping as a more powerful alternative to APIs. The
reverse is true: APIs are more powerful, more stable, and more efficient.
Scraping is what is done when the easier path is unavailable.

### Example 2 · Verifying the API Alternative
  
The same site that exposes the Associations, Certificates, and Conferences
tables also exposes a WordPress REST API. The cell below retrieves the API
response and prints the title and link of every published page. This is the
API-first principle made concrete on the very site about to be scraped.

In [ ]:
response = requests.get("https://datarollup.io/wp-json/wp/v2/pages",
                        headers=HEADERS, timeout=10)
response.raise_for_status()
pages = response.json()

for page in pages:
    print(page["title"]["rendered"], page["link"])

## Topic 1.2 · HTTP and HTML Preliminaries

Every web page is a response to an HTTP request. The request-response model
introduced in the Web APIs module applies unchanged. The only difference is
that the response body is HTML rather than JSON. HTML is a tree-structured
document where elements have a tag name, optional attributes, and either text
content or nested child elements.

In a browser, navigate to `https://datarollup.io/associations/`, right-click on
the table and choose **Inspect**. The developer tools panel shows the
underlying HTML tree. A `<tr>` element contains `<td>` children. This is the
structure that BeautifulSoup will navigate.

### Key Terms

- **Element:** a single unit of HTML structure such as `<table>`, `<tr>`, or `<a>`.
- **Tag:** the syntactic marker that opens or closes an element. The opening tag may include attributes.
- **Attribute:** a key-value pair attached to an element, such as `class="routine"` or `href="https://example.com"`.
- **Text node:** the human-readable text content of an element.
- **Document Object Model (DOM):** the in-memory tree structure that browsers build from HTML.

## Topic 1.3 · The `requests` Library

The `requests` library performs HTTP requests in Python with a clean, predictable
interface. The same library used in the API module is used here. The pattern
for retrieving a page is:

1. Construct a headers dictionary that always includes a `User-Agent`.
2. Call `requests.get()` with the URL and headers and a timeout.
3. Check for HTTP errors with `response.raise_for_status()`.
4. Access the response body via `response.text` for HTML or `response.json()` for JSON.

### Pattern A · The fetch

The cell below demonstrates Pattern A on the Associations page. The HTML body
is captured into the variable `html` for use in the BeautifulSoup section that
follows.

In [ ]:
response = requests.get("https://datarollup.io/associations/",
                        headers=HEADERS, timeout=10)
response.raise_for_status()
html = response.text

print(f"Status code: {response.status_code}")
print(f"Response length: {len(html):,} characters")
print(f"First 200 characters of body:")
print(html[:200])

### Why Headers Matter

A request without a `User-Agent` header sends the urllib default
(`Python-urllib/3.x`), which Cloudflare and similar bot-detection systems treat
as suspicious. The `User-Agent` header is the courteous way to identify a real
client with a legitimate purpose. The string includes a contact email so any
administrator who reviews logs can reach the author.

### Key Terms

- **HTTP header:** metadata sent with a request or response, separate from the body.
- **User-Agent:** the header that identifies the client making the request.
- **Status code:** the integer in the HTTP response indicating success (200), redirection (3xx), client error (4xx), or server error (5xx).
- **Timeout:** the maximum time `requests` waits for a response before giving up; ten seconds is a reasonable default.

## Topic 1.4 · BeautifulSoup Essentials

BeautifulSoup parses an HTML string into a navigable Python object. The four
most common operations cover almost all introductory scraping needs:

- `find()` retrieves the first matching element.
- `find_all()` retrieves a list of all matches.
- Attribute access via dictionary syntax retrieves attribute values.
- `.get_text()` returns the visible text content with surrounding whitespace removed.

### Pattern B · Manual BeautifulSoup parsing

The cell below demonstrates Pattern B on a small inline HTML string. Working
with an inline string first removes any network dependency and lets the parser
behavior be observed in isolation before applying it to a real page.

In [ ]:
sample_html = """
<html>
  <body>
    <h1>Sample Page</h1>
    <p>Some introductory text.</p>
    <ul>
      <li><a href="https://www.baruch.cuny.edu">Baruch College</a></li>
      <li><a href="https://zicklin.baruch.cuny.edu">Zicklin School of Business</a></li>
      <li><a href="https://www.cuny.edu">CUNY</a></li>
    </ul>
  </body>
</html>
"""

soup = BeautifulSoup(sample_html, "html.parser")

# Find the first link
first_link = soup.find("a")
print("First link href:", first_link.get("href"))
print("First link text:", first_link.get_text())
print()

# Find all links
all_links = soup.find_all("a")
print(f"Total links found: {len(all_links)}")
for link in all_links:
    print(f"  {link.get_text()} -> {link.get('href')}")

<font color=blue>💡 Tip: If you copy the contents of the docstring above without the triple-quotes and paste it into a text file with the extension `.htm`, you will have a web page that you can view in a browser. -- <i>be sure to start with the `<html>` tag and include the ending `</html>` tag. </i></font>

### CSS Selectors as an Alternative

The `.select()` method accepts CSS selector syntax for more concise queries.
For students familiar with CSS, this is a cleaner path. For students who are
not, `find_all()` with keyword arguments is the better starting point. Both are
shown; CSS selectors are the second, more advanced syntax.

In [ ]:
# CSS selector form for the same query
links_via_select = soup.select("ul li a")
print(f"Links found via .select(): {len(links_via_select)}")
for link in links_via_select:
    print(f"  {link.get_text()} -> {link.get('href')}")

### Do Not Use Regex on HTML

Regular expressions are not a reasonable way to parse HTML. HTML is not a
regular language, and regex extraction breaks on edge cases that real pages
routinely contain. BeautifulSoup exists precisely to solve this problem
properly. From this module forward, BeautifulSoup is the  parser
whenever HTML must be parsed.

### Key Terms

- **Parser:** the component that converts HTML text into a navigable structure. The standard library parser `html.parser` is used throughout this module.
- **Soup object:** the root of the parsed document tree.
- **CSS selector:** a syntax for identifying elements by tag, class, and attribute, originally developed for CSS styling.

## Topic 1.5 · The Dr. Chuck Severance Demonstration

A complete, minimal scrape using the simplest possible HTML.
[Dr. Chuck Page 1](http://www.dr-chuck.com/page1.htm) contains exactly one
anchor tag pointing to [Page 2](http://www.dr-chuck.com/page2.htm). Page 2
contains exactly one anchor tag pointing back to Page 1. These pages have
served the Coursera Python for Everybody course for years and are intentionally
simple enough that the entire HTML can be read in the notebook output.

These pages are external, stable, and pedagogically minimal: no JavaScript, no
advertising clutter, no responsive design noise. The simplicity is the lesson.  
  

### Example 1 · A Minimal Scrape

Fetch the simplest possible page, parse it with BeautifulSoup, and print every
link.

#### Viewing The Target Site
  
Do you remember Dr. Chuck's web pages `page1.htm` and `page2.htm` from the Python For Everybody lesson on networks?  You can view the website `http://www.dr-chuck.com/page1.htm` with your browser by double-clicking on the URL or typing the URL into your search bar.

```
http://www.dr-chuck.com/page1.htm
```

In [ ]:
response = requests.get("http://www.dr-chuck.com/page1.htm",
                        headers=HEADERS, timeout=10)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

print("Full HTML of Page 1:")
print(response.text)
print()
print("All links found on Page 1:")
for link in soup.find_all("a"):
    print(link.get("href"))

### Class 1 Exercise · Follow Page 1 to Page 2

The Dr. Chuck pages illustrate the fundamental scrape-extract-follow pattern.
The exercise below walks through it step by step.

**Task.** Retrieve Page 1, extract the URL of the single outbound link,
retrieve the page that link points to, and print the text content of the
heading on the second page.

**Implementation TODOs.** Complete the function below. Numbered TODO markers
indicate where each step belongs. The function returns the heading text from
Page 2 so the result can be inspected.

In [ ]:
def follow_link_and_get_heading(start_url):
    """
    Fetch start_url, parse it, follow the first <a> tag found, and return the
    text of the first <h1> on the destination page.

    Parameters
    ----------
    start_url : str
        The URL of the starting page.

    Returns
    -------
    str
        The text content of the first <h1> element on the destination page.
    """

    # TODO 1: Fetch the starting page using requests.get with HEADERS and a
    #         10-second timeout. Call response.raise_for_status() to surface
    #         any HTTP error.

    # TODO 2: Parse the response body with BeautifulSoup using "html.parser".

    # TODO 3: Find the first anchor tag (<a>) on the page and read its href
    #         attribute into a variable named next_url.
    #         Hint: soup.find("a") returns the first match; .get("href")
    #         reads the attribute.

    # TODO 4: Fetch next_url with the same headers-and-timeout pattern as
    #         step 1. Parse the new response body into a new soup object.

    # TODO 5: Find the first <h1> element on the destination page and return
    #         its text content. Use .get_text() and strip surrounding
    #         whitespace.

    pass  # Replace with your implementation


# Test the function
heading = follow_link_and_get_heading("http://www.dr-chuck.com/page1.htm")
print("Heading found on the destination page:", heading)

---

# Class 2 · Tabular Extraction and Defensive Practice

Class 2 introduces `pandas.read_html()` for one-line tabular extraction,
defensive practices for handling the irregularities that occur in real-world
tables, and a capstone exercise that combines fetch, parse, normalize, and
analyze on the Conferences table.

## Topic 2.1 · Tables and `pandas.read_html()`

When the structured data on a page is presented in an HTML `<table>` element,
the entire extraction can be performed in a single line: `pandas.read_html()`
retrieves the HTML, identifies all `<table>` elements, and returns each as a
Pandas DataFrame in a list. This is dramatically more concise than the manual
BeautifulSoup approach when the data is genuinely tabular, and it is the right
tool for the genuinely tabular case.

### Pattern C · Tabular extraction (production form)

The cell below demonstrates Pattern C on the Associations page. Note the
fetch-then-parse structure: the fetch is performed explicitly with appropriate
headers, and the resulting HTML text is wrapped in `StringIO` for `read_html()`.

In [ ]:
response = requests.get("https://datarollup.io/associations/",
                        headers=HEADERS, timeout=10)
response.raise_for_status()
tables = pd.read_html(StringIO(response.text))
df = tables[0]

print(f"Number of tables found on the page: {len(tables)}")
print(f"Shape of the first table: {df.shape}")
print()
print("First five rows:")
print(df.head())
print()
print("Column dtypes:")
print(df.dtypes)

In [ ]:
df

### Why Fetch Separately

The convenience form `pd.read_html(url)` works for friendly sites without bot
protection but fails on any site that requires identified clients. The
fetch-then-parse pattern is the production-quality approach: it gives the
developer control over headers, timeouts, and error handling. Class 1
established the fetch pattern; Class 2 reuses it and pipes the result to
`read_html`.

### Why `StringIO`

Pandas 2.x deprecated passing raw HTML strings directly to `read_html()`. The
replacement is to wrap the string in `io.StringIO`, which presents it as a
file-like object. This is a small ceremony that appears in any modern code.

### Key Terms

- **DataFrame:** the two-dimensional Pandas data structure used since earlier modules.
- **Type coercion:** the process by which Pandas infers the best dtype (`int`, `float`, string) for each column.
- **Object dtype:** the Pandas dtype for columns containing mixed types or arbitrary Python objects, often where strings appear.

## Topic 2.2 · Defensive Extraction Against Real-World Tables

Real tables contain irregularities that break naive extraction. The
`datarollup.io` tables are deliberately designed with such irregularities to
teach students to anticipate and handle them. A defensive extraction pipeline
checks for missing cells, normalizes string formats before numeric conversion,
and handles special-string tokens explicitly.

Three concrete examples appear in the class data and recur throughout the module:

- The **Associations** table contains `"Not disclosed"` in the *Members* column for the EDM Council and the suffix-laden `"30,000+"` for IIBA, both of which prevent native numeric coercion.
- The **Certificates** table contains `"Lifetime"` in the *Validity Years* column for ECBA and an em-dash (`"—"`) in the *Required Experience Years* column for ECBA, both of which are semantic placeholders that must be interpreted rather than parsed.
- The **Conferences** table contains `"Free"` in the *Registration USD* column for Big Data LDN, a categorical token in an otherwise currency-formatted column.

The cell below inspects the Associations table values that illustrate the
first two cases. Observe that the *Members* column is `object` dtype because
of the mixed string and numeric content.

In [ ]:
# The Associations table has been loaded above into df.
print("Members column dtype:", df["Members"].dtype)
print()
print("Unique values in the Members column:")
for value in df["Members"]:
    print(f"  {value!r}  (type: {type(value).__name__})")

###  Defensive Handling

The function below is the  defensive-extraction pattern. It type-checks
the input, normalizes the string by removing thousands separators and the
trailing `+`, recognizes the `"Not disclosed"` and `"—"` special tokens, and
attempts numeric conversion with explicit failure handling.

In [ ]:
def parse_members(value):
    """Convert the Members column to int where possible, NaN otherwise."""
    if pd.isna(value) or not isinstance(value, str):
        return value if isinstance(value, (int, float)) else None
    cleaned = value.replace(",", "").replace("+", "").strip()
    if cleaned.lower() in {"not disclosed", "—", ""}:
        return None
    try:
        return int(cleaned)
    except ValueError:
        return None


df["members_parsed"] = df["Members"].apply(parse_members)
print(df[["Members", "members_parsed"]])

### Class 2 Exercise A · Defensive Parser for Validity Years

The Certificates table contains the special token `"Lifetime"` in the *Validity
Years* column for the ECBA certificate. A naive numeric coercion of this column
would either fail or produce NaN for that row.

**Task.** Write a defensive parser that converts the *Validity Years* column to
a numeric column. The parser must treat `"Lifetime"` as a domain-specific
sentinel value and return a large numeric placeholder (use `999`) so the
"effectively unlimited" semantics survive numeric aggregation. All other
non-numeric values should return `None`.

**Implementation TODOs.** Complete the function below. The Certificates table
will be loaded into `df_certs` for the exercise.

In [ ]:
# Load the Certificates table for this exercise
response_c = requests.get("https://datarollup.io/certificates/",
                          headers=HEADERS, timeout=10)
response_c.raise_for_status()
df_certs = pd.read_html(StringIO(response_c.text))[0]

print(f"Certificates table shape: {df_certs.shape}")
print()
print(df_certs[["Certification", "Validity Years"]])

In [ ]:
def parse_validity_years(value):
    """
    Convert a Validity Years cell to a numeric value.

    Rules:
      - If the value is already numeric (int or float, including NaN),
        return it unchanged.
      - If the value is the string "Lifetime" (case-insensitive), return 999
        as a sentinel meaning "effectively unlimited."
      - If the value is the em-dash "—" or an empty string, return None.
      - Otherwise, attempt to convert the string to int; on failure, return None.
    """

    # TODO 1: Handle the case where the value is already numeric (int or float).
    #         Use isinstance(value, (int, float)). Return the value unchanged.
    #         Note: pd.isna(NaN) is True, but NaN is technically a float, so
    #         this branch handles NaN correctly without a separate check.

    # TODO 2: If the value is not a string at this point, return None as a
    #         safe default for unexpected types.

    # TODO 3: Normalize the string: strip whitespace and lowercase it for
    #         comparison.

    # TODO 4: If the normalized value equals "lifetime", return 999.

    # TODO 5: If the normalized value is in {"—", ""}, return None.

    # TODO 6: Attempt int() conversion on the original (non-lowercased) value
    #         after stripping whitespace. On ValueError, return None.

    pass  # Replace with your implementation


df_certs["validity_parsed"] = df_certs["Validity Years"].apply(parse_validity_years)
print(df_certs[["Certification", "Validity Years", "validity_parsed"]])

## Topic 2.3 · Connection to Pandas Operations

Once extracted, the DataFrame is operated on with the same techniques learned
in earlier modules. Scraping is not a separate skill; it is a data-acquisition
step that feeds existing analytical patterns.

The cell below loads the Conferences table and answers three sample questions
that use only Pandas techniques the student already knows from earlier modules.
The new contribution is that the data came from a web page rather than a CSV.

In [ ]:
# Load the Conferences table
response_conf = requests.get("https://datarollup.io/conferences/",
                             headers=HEADERS, timeout=10)
response_conf.raise_for_status()
df_conf = pd.read_html(StringIO(response_conf.text))[0]

print(f"Conferences table shape: {df_conf.shape}")
print()
print(df_conf.head())
print()
print("Column names:", list(df_conf.columns))

In [ ]:
# Question 1: How many conferences are organized by IIBA?
iiba_count = df_conf["Organization"].str.contains("IIBA").sum()
print(f"Conferences organized by IIBA: {iiba_count}")

# Question 2: What is the median registration fee, treating "Free" as missing?
median_fee = pd.to_numeric(
    df_conf["Registration USD"].str.replace("$", "").str.replace(",", ""),
    errors="coerce"
).median()
print(f"Median registration fee (Free treated as NaN): ${median_fee:.2f}")

# Question 3: How many conferences run in October?
october_count = df_conf["Annual Months"].str.contains("October").sum()
print(f"Conferences running in October: {october_count}")

## Topic 2.4 · The Capstone Exercise

The capstone is a complete extract-and-analyze pipeline on the Conferences
table. The pipeline structure is the  scraping shape: identify the
URL, retrieve with appropriate headers, parse to a DataFrame, normalize the
column of interest, and produce a small grouped analysis.

The capstone is split across two exercises below: the first writes the
defensive parser for the *Registration USD* column, and the second produces
the grouped analysis that consumes the parser output.

### Step 1 of 2 · Normalize the Registration USD Column

**Task.** Write a defensive parser for the *Registration USD* column. The
column contains conventional currency strings such as `"$1,995"` and a
categorical token `"Free"` for Big Data LDN. The parser must recognize `"Free"`
as zero, parse standard currency strings to float, and return `None` for any
other unrecognized string.

**Implementation TODOs.** Complete the function below.

In [ ]:
def parse_registration(value):
    """
    Convert a Registration USD cell to a numeric value.

    Rules:
      - Non-string values pass through unchanged.
      - "Free" (case-insensitive, stripped) returns 0.0.
      - Standard currency strings are stripped of "$" and "," and parsed to float.
      - Any other value returns None.
    """

    # TODO 1: If the value is not a string, return it unchanged. This handles
    #         numeric cells and NaN cells without further processing.

    # TODO 2: Normalize the string by stripping whitespace and lowercasing,
    #         then check whether it equals "free". If so, return 0.0.

    # TODO 3: Strip the dollar sign, comma, and surrounding whitespace from
    #         the original value to produce a cleaned numeric-string candidate.

    # TODO 4: Attempt float() conversion on the cleaned string. On ValueError,
    #         return None.

    pass  # Replace with your implementation


df_conf["registration_usd"] = df_conf["Registration USD"].apply(parse_registration)
print(df_conf[["Conference", "Registration USD", "registration_usd"]])

### Step 2 of 2 · Grouped Analysis

**Task.** Produce a grouped summary of registration fees by *Format*. For each
format value (Hybrid, In-Person, Virtual, or whatever values appear in the
loaded data), report the count, mean, minimum, and maximum of `registration_usd`.

**Implementation TODOs.** Complete the cell below.

In [ ]:
# TODO 1: Group df_conf by the "Format" column and aggregate the
#         "registration_usd" column with count, mean, min, and max.
#         Use the .agg() method with a list of aggregation names.

# TODO 2: Print the resulting summary DataFrame.

# Replace with your implementation
summary = None
print(summary)

## Topic 2.5 · Boundary Discussion · What Scraping Cannot Do

Static scraping has limits. JavaScript-rendered single-page applications,
content loaded through XHR after page load, and sites requiring authentication
all sit outside what `requests` plus BeautifulSoup can do. The next-step tools
are Selenium and Playwright, which drive a full browser and observe the
rendered DOM after JavaScript executes. These tools are named here without
being taught; they are the natural follow-up when static scraping fails.

This boundary is best framed as a return to the API-first principle: when
scraping becomes hard, the underlying API call powering the page is often the
better target. A page that fetches its data through JavaScript is making an
HTTP request to some endpoint; that endpoint is frequently a JSON API that can
be called directly with `requests`. The browser developer tools *Network* tab
reveals these underlying requests. When such a request is visible, the
API-first path has been rediscovered.

---

## Wrap-Up

Module 16 introduced web scraping as the fallback technique for extracting
structured data from web pages when no Web API is available. The module
established three  patterns that recur in any scraping work:

1. **Pattern A — The fetch.** `requests.get(url, headers=HEADERS, timeout=10)` followed by `response.raise_for_status()`.
2. **Pattern B — Manual BeautifulSoup parsing.** `BeautifulSoup(html, "html.parser")` with `find()`, `find_all()`, and `.get_text()`.
3. **Pattern C — Tabular extraction.** `pd.read_html(StringIO(response.text))[0]` for one-line DataFrame construction.

Defensive extraction was introduced as a first-class topic. Real tables
contain `"Not disclosed"`, `"30,000+"`, `"Lifetime"`, em-dashes, and `"Free"`
tokens that break naive numeric coercion. The defensive shape — type-check,
normalize, recognize special tokens, attempt conversion — is portable to any
column with mixed-type values.

The module deliberately reinforced the API-first principle. The same site that
served the scraping targets also exposed a complete WordPress REST API at
`/wp-json/wp/v2/pages`. The thoughtful default is to check for the API first;
scraping is the appropriate tool when the API is absent or insufficient.